In [1]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

import sys

sys.path.append("..")

In [2]:
from dataclasses import asdict

import dotenv

import wandb
from src.dataloaders import EnsembleSeasonAverageDataLoader, SeasonAverageDataLoader
from src.evaluation.evaluate import compute_brier_score_for_predictions, generate_matchups
from src.experiments import DefaultTracker, ExperimentConfig, WandbTracker
from src.experiments.config import get_run_name
from src.models.cross_validation import CrossValidationConfig
from src.models.xgboost import EnsembleXGBRegressorModel, XGBHyperparamConfig, XGBRegressorModel
from src.submissions.submission import create_submission

In [3]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: schurtenberger-david (david-schurtenberger) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Example training

In [3]:
features = [
    "MenWomen",
    "QualityDiff",
    # 'Season',
    "SeedDiff",
    "T1_Elo",
    "T1_Quality",
    # 'T1_TeamID',
    # 'T1_avg_Ast',
    "T1_avg_Blk",
    "T1_avg_DR",
    # 'T1_avg_Elo',
    # 'T1_avg_EloDelta',
    # 'T1_avg_EloDeltaDiff',
    # 'T1_avg_EloDiff',
    "T1_avg_FGA",
    # 'T1_avg_FGA3',
    # 'T1_avg_FGM',
    # 'T1_avg_FGM3',
    # 'T1_avg_FTA',
    # 'T1_avg_FTM',
    "T1_avg_OR",
    "T1_avg_PF",
    "T1_avg_PointDiff",
    "T1_avg_Score",
    # 'T1_avg_Stl',
    # 'T1_avg_Streak',
    # 'T1_avg_StreakDiff',
    # 'T1_avg_TO',
    # 'T1_avg_opponent_Ast',
    "T1_avg_opponent_Blk",
    # 'T1_avg_opponent_DR',
    # 'T1_avg_opponent_Elo',
    # 'T1_avg_opponent_EloDelta',
    "T1_avg_opponent_FGA",
    # 'T1_avg_opponent_FGA3',
    # 'T1_avg_opponent_FGM',
    # 'T1_avg_opponent_FGM3',
    # 'T1_avg_opponent_FTA',
    # 'T1_avg_opponent_FTM',
    # 'T1_avg_opponent_OR',
    "T1_avg_opponent_PF",
    # 'T1_avg_opponent_Score',
    # 'T1_avg_opponent_Stl',
    # 'T1_avg_opponent_Streak',
    # 'T1_avg_opponent_TO',
    "T1_seed",
    "T2_Elo",
    "T2_Quality",
    # 'T2_TeamID',
    # 'T2_avg_Ast',
    "T2_avg_Blk",
    "T2_avg_DR",
    # 'T2_avg_Elo',
    # 'T2_avg_EloDelta',
    # 'T2_avg_EloDeltaDiff',
    # 'T2_avg_EloDiff',
    "T2_avg_FGA",
    # 'T2_avg_FGA3',
    # 'T2_avg_FGM',
    # 'T2_avg_FGM3',
    # 'T2_avg_FTA',
    # 'T2_avg_FTM',
    "T2_avg_OR",
    "T2_avg_PF",
    "T2_avg_PointDiff",
    "T2_avg_Score",
    # 'T2_avg_Stl',
    # 'T2_avg_Streak',
    # 'T2_avg_StreakDiff',
    # 'T2_avg_TO',
    # 'T2_avg_opponent_Ast',
    "T2_avg_opponent_Blk",
    # 'T2_avg_opponent_DR',
    # 'T2_avg_opponent_Elo',
    # 'T2_avg_opponent_EloDelta',
    "T2_avg_opponent_FGA",
    # 'T2_avg_opponent_FGA3',
    # 'T2_avg_opponent_FGM',
    # 'T2_avg_opponent_FGM3',
    # 'T2_avg_opponent_FTA',
    # 'T2_avg_opponent_FTM',
    # 'T2_avg_opponent_OR',
    "T2_avg_opponent_PF",
    # 'T2_avg_opponent_Score',
    # 'T2_avg_opponent_Stl',
    # 'T2_avg_opponent_Streak',
    # 'T2_avg_opponent_TO',
    "T2_seed",
]

In [5]:
validation_season = 2024
data_loader = SeasonAverageDataLoader(features)
regressor_config = XGBHyperparamConfig()
experiment_config = ExperimentConfig(name=get_run_name(regressor_config, features, validation_season))
tracker = WandbTracker(experiment_config, asdict(regressor_config))

In [6]:
tracker.start()

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [7]:
model = XGBRegressorModel(data_loader, regressor_config, CrossValidationConfig(), tracker)

In [8]:
model.fit(season=validation_season)

In [9]:
model.validate()

In [ ]:
tracker.finish()

In [11]:
df_matchup = generate_matchups(validation_season)
df_matchup["Pred"] = model.predict(df_matchup.copy())
compute_brier_score_for_predictions(df_matchup, validation_season)

np.float64(0.1637672244500258)

## Create example submission for intantiated model

In [4]:
create_submission(2025, model, fit=True)

NameError: name 'model' is not defined

### Test Ensemble Model

In [10]:
test_season = 2025
data_loader = EnsembleSeasonAverageDataLoader(0)
regressor_config = XGBHyperparamConfig()
experiment_config = ExperimentConfig(name="Test")
tracker = DefaultTracker({})

In [11]:
model = EnsembleXGBRegressorModel(data_loader, regressor_config, tracker)

In [12]:
model.fit(test_season=test_season)

metrics: {'season': 2003, 'train_brier_ensemble': np.float64(0.09121623944943036)}, step: None
metrics: {'season': 2003, 'valid_brier_ensemble': np.float64(0.19024458192938898)}, step: None
metrics: {'season': 2004, 'train_brier_ensemble': np.float64(0.09197246938455442)}, step: None
metrics: {'season': 2004, 'valid_brier_ensemble': np.float64(0.17420736601307032)}, step: None
metrics: {'season': 2005, 'train_brier_ensemble': np.float64(0.09268343881888805)}, step: None
metrics: {'season': 2005, 'valid_brier_ensemble': np.float64(0.1748792509911732)}, step: None
metrics: {'season': 2006, 'train_brier_ensemble': np.float64(0.09192141598620533)}, step: None
metrics: {'season': 2006, 'valid_brier_ensemble': np.float64(0.2006692935753739)}, step: None
metrics: {'season': 2007, 'train_brier_ensemble': np.float64(0.09280474437585204)}, step: None
metrics: {'season': 2007, 'valid_brier_ensemble': np.float64(0.16600242979886565)}, step: None
metrics: {'season': 2008, 'train_brier_ensemble': np